# Tensors and Autograd

Companion to `notes.md`. We build intuition for reverse-mode automatic differentiation with a tiny
worked example, then run the **exact same 2-layer XOR MLP** as
`06-deep-learning/01-ann/ann-from-scratch-xor.ipynb` through PyTorch's autograd instead of hand-written
backprop, and numerically cross-check the two against each other.

In [1]:
import numpy as np
import torch

np.random.seed(0)
torch.manual_seed(0)
np.set_printoptions(precision=6, suppress=True)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

torch: 2.13.0+cpu
cuda available: False


## 1. A tiny worked example: `.backward()` vs. a hand-computed derivative

$L = (\sin(x_1 x_2) + x_2)^2$, differentiated by hand in `notes.md`'s "Intuition" section:

```
a = x1 * x2
b = sin(a)
c = b + x2
L = c ** 2
```

Build the same computation with `requires_grad=True` tensors, call `.backward()`, and compare
`.grad` against the hand-derived chain-rule product.

In [2]:
x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)

a = x1 * x2
b = torch.sin(a)
c = b + x2
L = c ** 2

L.backward()

print("L =", L.item())
print("autograd dL/dx1 =", x1.grad.item())
print("autograd dL/dx2 =", x2.grad.item())

L = 7.4015793800354
autograd dL/dx1 = 15.673345565795898
autograd dL/dx2 = 15.890066146850586


In [3]:
import math

# Hand-derived via the chain rule (notes.md "Intuition"):
# dL/dc = 2c ; dc/db = 1 ; db/da = cos(a) ; da/dx1 = x2 ; da/dx2 = x1 ; dc/dx2 = 1 (direct edge)
a_val = 2.0 * 3.0
c_val = math.sin(a_val) + 3.0
dL_dc = 2 * c_val
dL_dx1_hand = dL_dc * 1.0 * math.cos(a_val) * 3.0
dL_dx2_hand = dL_dc * 1.0 * math.cos(a_val) * 2.0 + dL_dc * 1.0

print("hand dL/dx1 =", dL_dx1_hand)
print("hand dL/dx2 =", dL_dx2_hand)
print("match:", np.allclose(x1.grad.item(), dL_dx1_hand), np.allclose(x2.grad.item(), dL_dx2_hand))

hand dL/dx1 = 15.673346405705281
hand dL/dx2 = 15.89006660740567
match: True True


`.backward()` reproduces the hand-derived chain-rule product exactly — this is the mechanism
`notes.md`'s "Mathematical foundation" describes generically. Now the real cross-check: the same
mechanism applied to the from-scratch MLP's actual backprop derivation.

## 2. `torch.Tensor` vs. `np.ndarray`: same data, tracked computation

`torch.Tensor` behaves like `np.ndarray` for shape/indexing/arithmetic, but with `requires_grad=True`
it additionally records every operation performed on it into a computation graph.

In [4]:
arr = np.array([1.0, 2.0, 3.0])
t = torch.tensor(arr, requires_grad=True)

print("np.ndarray:", arr, type(arr))
print("torch.Tensor:", t, type(t))
print("same underlying values:", np.allclose(arr, t.detach().numpy()))

np.ndarray: [1. 2. 3.] <class 'numpy.ndarray'>
torch.Tensor: tensor([1., 2., 3.], dtype=torch.float64, requires_grad=True) <class 'torch.Tensor'>
same underlying values: True


## 3. The from-scratch XOR MLP's forward pass, loss, and manual backward pass

Reproduced from `06-deep-learning/01-ann/ann-from-scratch-xor.ipynb` verbatim (same architecture,
same seed) — this is **not** re-derived, only re-run here so its gradients can be compared directly
against PyTorch's autograd gradients on identical weights and data.

In [5]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
], dtype=float)
y = np.array([0, 1, 1, 0], dtype=float).reshape(-1, 1)  # XOR

n_input, n_hidden, n_output = 2, 4, 1
np.random.seed(0)
W1 = np.random.randn(n_input, n_hidden) * 0.5
b1 = np.zeros((1, n_hidden))
W2 = np.random.randn(n_hidden, n_output) * 0.5
b2 = np.zeros((1, n_output))

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    a1 = np.tanh(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)
    return a2, (z1, a1, z2, a2)

def bce_loss(y, yhat):
    eps = 1e-9
    return -np.mean(y * np.log(yhat + eps) + (1 - y) * np.log(1 - yhat + eps))

def backward(X, y, cache, W1, W2):
    # exactly ann-from-scratch-xor.ipynb's backward()
    m = X.shape[0]
    z1, a1, z2, a2 = cache
    delta2 = a2 - y
    dW2 = a1.T @ delta2 / m
    db2 = np.sum(delta2, axis=0, keepdims=True) / m
    dtanh = 1 - np.tanh(z1) ** 2
    delta1 = (delta2 @ W2.T) * dtanh
    dW1 = X.T @ delta1 / m
    db1 = np.sum(delta1, axis=0, keepdims=True) / m
    return dW1, db1, dW2, db2

yhat, cache = forward(X, W1, b1, W2, b2)
manual_loss = bce_loss(y, yhat)
dW1, db1, dW2, db2 = backward(X, y, cache, W1, W2)

print("manual (from-scratch) loss:", manual_loss)
print("dW1:\n", dW1)
print("db1:\n", db1)
print("dW2:\n", dW2)
print("db2:\n", db2)

manual (from-scratch) loss: 0.7195357237154479
dW1:
 [[ 0.001437  0.012378  0.000005  0.022546]
 [ 0.002409  0.007626 -0.002805 -0.052725]]
db1:
 [[-0.001793  0.016081  0.001209  0.016124]]
dW2:
 [[-0.011899]
 [-0.00016 ]
 [ 0.017645]
 [ 0.06171 ]]
db2:
 [[0.059628]]


## 4. The identical computation via PyTorch autograd — no hand-written `backward()`

Same weight *values* (copied from the NumPy arrays above, not re-randomized), same forward equations,
but the gradient comes entirely from `.backward()` — `dW1, db1, dW2, db2` are never derived or coded
by hand here.

In [6]:
Xt = torch.tensor(X, dtype=torch.float64)
yt = torch.tensor(y, dtype=torch.float64)
W1t = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
b1t = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
W2t = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
b2t = torch.tensor(b2, dtype=torch.float64, requires_grad=True)

z1t = Xt @ W1t + b1t
a1t = torch.tanh(z1t)
z2t = a1t @ W2t + b2t
a2t = torch.sigmoid(z2t)

loss_t = torch.nn.functional.binary_cross_entropy(a2t, yt)
print("autograd (PyTorch) loss:", loss_t.item())

loss_t.backward()  # <-- this one call replaces the entire hand-written backward() above

print("W1.grad:\n", W1t.grad.numpy())
print("b1.grad:\n", b1t.grad.numpy())
print("W2.grad:\n", W2t.grad.numpy())
print("b2.grad:\n", b2t.grad.numpy())

autograd (PyTorch) loss: 0.7195357258090367
W1.grad:
 [[ 0.001437  0.012378  0.000005  0.022546]
 [ 0.002409  0.007626 -0.002805 -0.052725]]
b1.grad:
 [[-0.001793  0.016081  0.001209  0.016124]]
W2.grad:
 [[-0.011899]
 [-0.00016 ]
 [ 0.017645]
 [ 0.06171 ]]
b2.grad:
 [[0.059628]]


## 5. Cross-check: autograd gradients vs. from-scratch manual gradients

**Hypothesis:** both compute reverse-mode differentiation of the identical graph at identical weight
values, so every gradient tensor should match to floating-point precision (`np.allclose` should be
`True` for all four).

In [7]:
diffs = {
    "dW1": np.abs(dW1 - W1t.grad.numpy()).max(),
    "db1": np.abs(db1 - b1t.grad.numpy()).max(),
    "dW2": np.abs(dW2 - W2t.grad.numpy()).max(),
    "db2": np.abs(db2 - b2t.grad.numpy()).max(),
}
for name, d in diffs.items():
    print(f"max |{name} diff| = {d}")

matches = {
    "dW1": np.allclose(dW1, W1t.grad.numpy()),
    "db1": np.allclose(db1, b1t.grad.numpy()),
    "dW2": np.allclose(dW2, W2t.grad.numpy()),
    "db2": np.allclose(db2, b2t.grad.numpy()),
}
for name, m in matches.items():
    print(f"np.allclose({name}, {name[1:]}.grad) = {m}")

all_match = all(matches.values())
print("\nALL GRADIENTS MATCH:", all_match)
assert all_match, "gradients do not match"

max |dW1 diff| = 6.938893903907228e-18
max |db1 diff| = 6.938893903907228e-18
max |dW2 diff| = 2.7755575615628914e-17
max |db2 diff| = 2.7755575615628914e-17
np.allclose(dW1, W1.grad) = True
np.allclose(db1, b1.grad) = True
np.allclose(dW2, W2.grad) = True
np.allclose(db2, b2.grad) = True

ALL GRADIENTS MATCH: True


## Takeaway

PyTorch's autograd, given the identical weights and identical forward computation, reproduces the
from-scratch notebook's hand-derived gradients exactly (to floating-point precision), without a single
line of hand-written backward-pass code. This confirms `.backward()` is not a different or approximate
method — it is the same reverse-mode chain-rule computation `ann-from-scratch-xor.ipynb`'s `backward()`
performs by hand, generalized so it never has to be re-derived for a new architecture (see `notes.md`'s
"From-scratch implementation" section for the line-by-line mapping).